In [1]:
import os
import glob
import time

import numpy as np
import pandas as pd

from prospecpy.baseline import (
    arpls_baseline_second_deriv_weights
)

In [2]:
spectrum_files = glob.glob(
    "output_plots/pH6/*/cut_subtracted_data.csv"
)

len(spectrum_files)

19

In [3]:
selected_parameters = {
    "lam": 1e5,
    "peak_window": 35,
    "peak_weight": 0.3,
    "alpha": 0.8
}

In [4]:
benchmark_results = []

In [5]:
for file in spectrum_files:

    # Load spectrum
    data = pd.read_csv(file)

    x = data["wavenumber"].values

    y = data["cut_subtracted_absorbance"].values


    # Load peak information
    peak_file = file.replace(
        "cut_subtracted_data.csv",
        "arpls_baseline_corrected_peak_info.csv"
    )

    peak_data = pd.read_csv(peak_file)

    peak_wavenumbers = (
        peak_data["wavenumber"].values
    )


    # Runtime measurement
    start_time = time.perf_counter()


    baseline = arpls_baseline_second_deriv_weights(
        y,
        x,
        peak_wavenumbers,
        lam=1e5,
        ratio=1e-6,
        max_iter=50,
        peak_window=35,
        peak_weight=0.3,
        alpha=0.8,
    )


    end_time = time.perf_counter()


    runtime = end_time - start_time


    benchmark_results.append(
        {
            "spectrum": os.path.basename(
                os.path.dirname(file)
            ),

            "runtime_seconds": runtime,

            "data_points": len(y),

            "success": (
                len(baseline) == len(y)
                and np.isfinite(baseline).all()
            )
        }
    )

In [6]:
benchmark_df = pd.DataFrame(
    benchmark_results
)

benchmark_df

,spectrum,runtime_seconds,data_points,success
0,011a as iso Hyd2 dark titration 0 mV.0022,0.017538,157,True
1,011h as iso Hyd2 dark titration -300mV.0006,0.009639,157,True
2,011q as iso Hyd2 dark titration -550mV.0012,0.016064,157,True
3,011d as iso Hyd2 dark titration -1050mV.0007,0.050990,157,True
4,011t as iso Hyd2 dark titration -800mV.0004,0.028663,157,True
5,011r as iso Hyd2 dark titration -600mV.0009,0.036521,157,True
6,011n as iso Hyd2 dark titration -450mV.0014,0.082449,157,True
7,011p as iso Hyd2 dark titration -500mV.0007,0.074857,157,True
8,011f as iso Hyd2 dark titration -250mV.0008,0.031209,157,True
9,011c as iso Hyd2 dark titration -100 mV.0003,0.010201,157,True


In [7]:
benchmark_summary = pd.DataFrame(
    {
        "Metric": [
            "Number of spectra",
            "Successful corrections",
            "Success rate",
            "Average runtime (s)",
            "Maximum runtime (s)"
        ],

        "Value": [
            len(benchmark_df),
            benchmark_df["success"].sum(),
            benchmark_df["success"].mean(),
            benchmark_df["runtime_seconds"].mean(),
            benchmark_df["runtime_seconds"].max()
        ]
    }
)

benchmark_summary

,Metric,Value
0,Number of spectra,19.000000
1,Successful corrections,19.000000
2,Success rate,1.000000
3,Average runtime (s),0.037165
4,Maximum runtime (s),0.082449


## Performance benchmark summary

The final peak-guided arPLS method was evaluated on 19 experimental
hydrogenase FTIR spectra using the selected parameters
(λ = 1e5, peak_weight = 0.3, peak_window = 35).

All 19 spectra were successfully processed without numerical failures,
achieving a 100% correction success rate. The runtime was recorded for
each spectrum, demonstrating efficient baseline correction within the
ProSpecPy workflow.